In [ ]:

import pandas as pd

# 数据路径
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/test.csv'

# 读取数据
train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

# 查看数据的前几行
train_data.head()


Displayed on the user's machine.

In [ ]:


# 检查缺失值
train_data.isnull().sum()



id                                0
Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64

In [ ]:


# 特征构建：计算BMI
train_data['BMI'] = train_data['Weight'] / (train_data['Height'] ** 2)
test_data['BMI'] = test_data['Weight'] / (test_data['Height'] ** 2)

# 查看训练集的前几行，确认新特征是否已添加
train_data.head()



Displayed on the user's machine.

In [ ]:



# 特征编码：将分类变量转换为数值型变量
train_data_encoded = pd.get_dummies(train_data, drop_first=True)
test_data_encoded = pd.get_dummies(test_data, drop_first=True)

# 查看编码后的训练集的前几行
train_data_encoded.head()




Displayed on the user's machine.

In [ ]:



from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split

# 分离特征和目标变量
X_train = train_data_encoded.drop(columns=['id', 'NObeyesdad'])
y_train = train_data_encoded['NObeyesdad']

# 初始化随机森林模型
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 训练模型
rf_model.fit(X_train, y_train)

# 特征重要性
feature_importances = rf_model.feature_importances_

# 选择重要特征
selector = SelectFromModel(rf_model, prefit=True)
X_train_selected = selector.transform(X_train)

# 查看选择的特征
selected_features = X_train.columns[selector.get_support()]
selected_features




---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
Cell In[6], line 12
     10 pass
     11 print('##active_line6##')
---> 12 X_train = train_data_encoded.drop(columns=['id', 'NObeyesdad'])
     13 print('##active_line7##')
     14 y_train = train_data_encoded['NObeyesdad']

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\frame.py:5581, in DataFrame.drop(self, labels, axis, index, columns, level, inplace, errors)
   5433 def drop(
   5434     self,
   5435     labels: IndexLabel | None = None,
   (...)
   5442     errors: IgnoreRaise = "raise",
   5443 ) -> DataFrame | None:
   5444     """
   5445     Drop specified labels from rows or columns.
   5446 
   (...)
   5579             weight  1.0     0.8
   5580     """
-> 5581     return super().drop(
   5582         labels=labels,
   5583         axis=axis,
   5584         index=index,

In [ ]:



# 分离特征和目标变量
X_train = train_data.drop(columns=['id', 'NObeyesdad'])
y_train = train_data['NObeyesdad']
X_test = test_data.drop(columns=['id', 'NObeyesdad'])
y_test = test_data['NObeyesdad']

# 特征编码：将分类变量转换为数值型变量
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# 确保训练集和测试集的特征列一致
X_test_encoded = X_test_encoded[X_train_encoded.columns]

# 查看编码后的训练集的前几行
X_train_encoded.head()




Displayed on the user's machine.

In [ ]:




from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

# 初始化随机森林模型
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 训练模型
rf_model.fit(X_train_encoded, y_train)

# 特征重要性
feature_importances = rf_model.feature_importances_

# 选择重要特征
selector = SelectFromModel(rf_model, prefit=True)
X_train_selected = selector.transform(X_train_encoded)
X_test_selected = selector.transform(X_test_encoded)

# 查看选择的特征
selected_features = X_train_encoded.columns[selector.get_support()]
selected_features





D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
Index(['Age', 'Height', 'Weight', 'FCVC', 'BMI', 'Gender_Male'], dtype='object')

In [ ]:



from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 使用选中的特征训练最终模型
final_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model.fit(X_train_selected, y_train)

# 在测试集上进行预测
y_pred = final_model.predict(X_test_selected)

# 评估模型性能
classification_report_result = classification_report(y_test, y_pred)
classification_report_result




'                     precision    recall  f1-score   support\n\nInsufficient_Weight       0.92      0.91      0.92       524\n      Normal_Weight       0.84      0.85      0.85       626\n     Obesity_Type_I       0.87      0.85      0.86       543\n    Obesity_Type_II       0.96      0.97      0.96       657\n   Obesity_Type_III       1.00      1.00      1.00       804\n Overweight_Level_I       0.72      0.74      0.73       484\nOverweight_Level_II       0.76      0.75      0.76       514\n\n           accuracy                           0.88      4152\n          macro avg       0.87      0.87      0.87      4152\n       weighted avg       0.88      0.88      0.88      4152\n'